# Orchestrator-Workers: Dynamic Task Decomposition

**What you'll learn:**
- How an orchestrator LLM dynamically breaks down tasks at runtime
- The critical difference from static parallelization
- XML-based coordination between orchestrator and workers
- ACI design principles applied to worker interfaces
- When dynamic decomposition justifies the overhead

**Position on the spectrum:** The most flexible workflow pattern — subtasks are determined per input, not predefined.

> *"The key difference from parallelization is flexibility — subtasks aren't pre-defined, but determined by the orchestrator based on the specific input."*

## How It Works

![Orchestrator-Workers workflow — Orchestrator dynamically delegates to worker LLM calls, Synthesizer combines results](assets/orchestrator_workers.webp)

**Two-phase architecture:**

### Phase 1: Analysis & Planning (Orchestrator)
The orchestrator LLM receives the task, analyzes what approaches would be most valuable, and generates structured subtask descriptions.

### Phase 2: Execution (Workers)
Each worker LLM receives the original task context + its specific subtask instructions. Workers operate independently.

**Why XML for coordination?**
- LLMs produce XML reliably (close to training data — ACI principle #2)
- No escaping issues unlike JSON (code inside JSON requires escaping quotes/newlines)
- Easy to parse with regex (no heavy dependencies)
- Gives the model "thinking space" before committing to structured output (ACI principle #3)

## When to Use Orchestrator-Workers

✅ **Use when:**
- Tasks require multiple distinct approaches or perspectives
- The optimal subtasks **depend on the specific input** (can't predefine)
- Different inputs need different decompositions
- You need to compare different strategies or styles

❌ **Don't use when:**
- Subtasks are always the same regardless of input (use [Parallelization](03_parallelization.ipynb))
- The task is linear/sequential (use [Prompt Chaining](01_prompt_chaining.ipynb))
- Latency is critical (orchestration adds an extra LLM call overhead)
- A single call handles the task adequately

### Cost: N+1 LLM calls minimum
- 1 orchestrator call (analysis + planning)
- N worker calls (one per identified subtask)
- Optional: 1 synthesis call to combine results

In [ ]:
import sys
sys.path.append(".")
from util import llm_call, extract_xml

In [ ]:
def parse_tasks(tasks_xml: str) -> list[dict]:
    """Parse the orchestrator's XML output into structured task dictionaries.
    
    Expected format:
    <task>
        <type>task_type</type>
        <description>what the worker should do</description>
    </task>
    """
    tasks = []
    current_task = {}

    for line in tasks_xml.split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("<task>"):
            current_task = {}
        elif line.startswith("<type>"):
            current_task["type"] = line[6:-7].strip()
        elif line.startswith("<description>"):
            current_task["description"] = line[13:-14].strip()
        elif line.startswith("</task>"):
            if "description" in current_task:
                if "type" not in current_task:
                    current_task["type"] = "default"
                tasks.append(current_task)

    return tasks

In [ ]:
class FlexibleOrchestrator:
    """Dynamically decompose tasks and delegate to specialized workers."""

    def __init__(self, orchestrator_prompt: str, worker_prompt: str):
        self.orchestrator_prompt = orchestrator_prompt
        self.worker_prompt = worker_prompt

    def process(self, task: str, context: dict | None = None) -> dict:
        """Process task by dynamically breaking it down and running subtasks."""
        context = context or {}

        # Phase 1: Orchestrator analyzes and plans
        orchestrator_input = self.orchestrator_prompt.format(task=task, **context)
        orchestrator_response = llm_call(orchestrator_input)

        analysis = extract_xml(orchestrator_response, "analysis")
        tasks_xml = extract_xml(orchestrator_response, "tasks")
        tasks = parse_tasks(tasks_xml)

        print("=" * 60)
        print("ORCHESTRATOR ANALYSIS")
        print("=" * 60)
        print(f"\n{analysis}\n")
        print(f"\nIdentified {len(tasks)} subtasks:")
        for i, t in enumerate(tasks, 1):
            print(f"  {i}. [{t['type']}] {t['description']}")

        # Phase 2: Workers execute subtasks
        print(f"\n{'=' * 60}")
        print("WORKER OUTPUTS")
        print("=" * 60)

        worker_results = []
        for i, task_info in enumerate(tasks, 1):
            print(f"\n[{i}/{len(tasks)}] Processing: {task_info['type']}...")

            worker_input = self.worker_prompt.format(
                original_task=task,
                task_type=task_info["type"],
                task_description=task_info["description"],
                **context,
            )

            worker_response = llm_call(worker_input)
            worker_content = extract_xml(worker_response, "response")

            if not worker_content or not worker_content.strip():
                worker_content = f"[Worker '{task_info['type']}' returned no content]"

            worker_results.append({
                "type": task_info["type"],
                "description": task_info["description"],
                "result": worker_content,
            })
            print(f"  Done ({len(worker_content)} chars)")

        # Display results
        print(f"\n{'=' * 60}")
        print("RESULTS")
        print("=" * 60)
        for i, result in enumerate(worker_results, 1):
            print(f"\n{'─' * 60}")
            print(f"  [{result['type'].upper()}]")
            print(f"{'─' * 60}")
            print(result['result'][:500])

        return {"analysis": analysis, "worker_results": worker_results}

## Example 1: Marketing Copy Variations

The orchestrator examines a product and decides which marketing angles are most valuable for THIS specific product — it might choose humor for a fun product, technical specs for a B2B tool, or sustainability for an eco product. The variations aren't hardcoded.

In [ ]:
ORCHESTRATOR_PROMPT = """Analyze this task and determine 2-3 distinct approaches that would be most valuable:

Task: {task}

Consider the target audience and product characteristics to choose the BEST angles 
(don't just use generic variations — be specific to this product).

<analysis>
Your understanding of what makes this product unique and which angles would resonate.
</analysis>

<tasks>
    <task>
    <type>approach_name</type>
    <description>Specific instructions for this variation</description>
    </task>
</tasks>"""

WORKER_PROMPT = """Generate content based on these instructions:

Original task: {original_task}
Your assigned style: {task_type}
Specific guidelines: {task_description}

Write compelling, ready-to-use copy. Be specific and vivid, not generic.

<response>
Your content here.
</response>"""

orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    worker_prompt=WORKER_PROMPT,
)

results = orchestrator.process(
    task="Write a product description for a new eco-friendly water bottle",
    context={
        "target_audience": "environmentally conscious millennials",
        "key_features": "plastic-free, insulated, lifetime warranty",
    },
)

## ACI Design in Practice

Let's examine how Agent-Computer Interface principles appear in this implementation:

| ACI Principle | How It's Applied Here |
|---------------|----------------------|
| **Empathize with the model** | Worker prompt includes original task context so the model understands the bigger picture |
| **Format for LLM strengths** | XML output format (not JSON) — no escaping issues with generated content |
| **Give thinking space** | `<analysis>` tag lets orchestrator reason before committing to task breakdown |
| **Poka-yoke** | Task structure requires both `type` and `description` — incomplete outputs get caught |
| **Document like onboarding** | Worker prompt is explicit: "Your assigned style: X, Guidelines: Y" |

### The Orchestrator's Quality is Your Ceiling

The decomposition step is the most critical. A poor orchestrator that generates vague or redundant subtasks will produce poor worker outputs regardless of worker quality. Invest prompt engineering effort here:

- **Bad:** `<type>version1</type>` — Worker doesn't know what makes this version distinct
- **Good:** `<type>sustainability-focused</type><description>Emphasize environmental impact, carbon footprint reduction, and materials sourcing. Use data points.</description>`

## Example 2: Multi-Perspective Code Review

A more technical use case: given code, the orchestrator identifies which review perspectives are most valuable (security? performance? readability?) based on what it sees in the code itself.

In [ ]:
CODE_REVIEW_ORCHESTRATOR = """Analyze this code and determine 2-3 review perspectives that would be most valuable.
Don't just use generic categories — look at the code and identify where the REAL risks and improvement opportunities are.

Task: Review this code
Code:
{task}

<analysis>
What you notice about this code and which review angles matter most.
</analysis>

<tasks>
    <task>
    <type>review_focus</type>
    <description>Specific aspects to evaluate and what to look for</description>
    </task>
</tasks>"""

CODE_REVIEW_WORKER = """You are a senior engineer performing a focused code review.

Code under review:
{original_task}

Your review focus: {task_type}
Specific instructions: {task_description}

Provide actionable feedback with specific line references and suggested fixes.

<response>
Your focused review here.
</response>"""

code_reviewer = FlexibleOrchestrator(
    orchestrator_prompt=CODE_REVIEW_ORCHESTRATOR,
    worker_prompt=CODE_REVIEW_WORKER,
)

sample_code = """
import sqlite3
import hashlib

def create_user(username, password, email):
    conn = sqlite3.connect('users.db')
    hashed = hashlib.md5(password.encode()).hexdigest()
    conn.execute(f"INSERT INTO users VALUES ('{username}', '{hashed}', '{email}')")
    conn.commit()
    conn.close()
    return True

def get_users(role):
    conn = sqlite3.connect('users.db')
    cursor = conn.execute(f"SELECT * FROM users WHERE role = '{role}'")
    users = cursor.fetchall()
    conn.close()
    return users

def send_welcome_email(user):
    import smtplib
    server = smtplib.SMTP('smtp.company.com')
    server.login('admin', 'password123')
    server.sendmail('noreply@company.com', user['email'], 'Welcome!')
    server.quit()
"""

results = code_reviewer.process(task=sample_code)

## Pitfalls

| Pitfall | Impact | Mitigation |
|---------|--------|------------|
| **Vague orchestrator output** | Workers produce generic, unfocused content | Invest in orchestrator prompt — it's the ceiling |
| **Too many subtasks** | Cost and latency explode (N+1 calls) | Limit to 2-4 subtasks in the orchestrator prompt |
| **Redundant subtasks** | Workers produce overlapping content | Tell orchestrator to ensure subtasks are "distinct and non-overlapping" |
| **No synthesis step** | Raw worker outputs lack coherence | Add a final synthesis LLM call to combine results |
| **Workers lack context** | Outputs don't align with original intent | Always pass original task to workers alongside subtask instructions |

### Orchestrator-Workers vs. Other Patterns

| Scenario | Better Pattern | Why |
|----------|---------------|-----|
| Subtasks are always the same | [Parallelization](03_parallelization.ipynb) | Simpler, no orchestration overhead |
| Subtasks are sequential | [Prompt Chaining](01_prompt_chaining.ipynb) | Order matters |
| Need iterative improvement | [Evaluator-Optimizer](05_evaluator_optimizer.ipynb) | Feedback loop, not decomposition |
| Need adaptive decomposition | **Orchestrator-Workers** | This pattern! |

## Key Takeaways

1. **Orchestrator-Workers = runtime decomposition** — the LLM decides what subtasks are needed
2. **XML coordination** is reliable and LLM-friendly (ACI principle)
3. **Orchestrator quality is the ceiling** — invest prompt engineering there
4. **Cost: N+1 calls minimum** — justify the overhead with better results
5. **Give workers full context** — original task + specific instructions

---

**Next up:** [05_evaluator_optimizer.ipynb](05_evaluator_optimizer.ipynb) — When you need iterative refinement, not decomposition.